In [4]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from torchvision import transforms
import matplotlib.pyplot as plt

In [5]:
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.GTSRB(
    root='./data',
    split='train',
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.GTSRB(
    root='./data',
    split='test',
    download=True,
    transform=transform
)

In [6]:
train_size = int(0.8*len(train_dataset))
val_size = len(train_dataset) - train_size

train_set, val_set = random_split(train_dataset,[train_size,val_size])

In [10]:
train_loader = DataLoader(train_set,batch_size=128,shuffle=True,num_workers=2)
val_loader = DataLoader(val_set,batch_size=128)
test_loader = DataLoader(test_dataset,batch_size=128)

In [11]:
class CNN(nn.Module):
    def __init__(self, num_classes=43, p=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(p)
        self.fc1 = nn.Linear(128*4*4,256)
        self.fc2 = nn.Linear(256,num_classes)

    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0),-1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x